# **Day 49: Object Oriented Programming (Part 9) - Operator Overloading**

**Operator overloading** means teaching Python's maths symbols (`+`, `-`, `*`, `/`) how to work with **our own objects**.

We do this by writing magic methods:

| Symbol | Method | Reverse method (for `number * object`) |
|---|---|---|
| `+` | `__add__` | `__radd__` |
| `-` | `__sub__` | `__rsub__` |
| `*` | `__mul__` | `__rmul__` |
| `/` | `__truediv__` | `__rtruediv__` |
| `-x` | `__neg__` | |
| `abs(x)` | `__abs__` | |

## **Build Custom Class: Fraction**

In [1]:
from math import gcd

# Create a 'Fraction' class to represent fractions and do arithmetic
class Fraction:

    # Constructor: store the numerator and denominator in lowest form
    def __init__(self, num, den=1):
        if den == 0:
            raise ZeroDivisionError("Denominator cannot be zero")
        g = gcd(num, den) * (1 if den > 0 else -1)
        self.num = num // g
        self.den = den // g

    # String representation, for example '3/4'
    def __str__(self):
        return f"{self.num}/{self.den}" if self.den != 1 else str(self.num)

    # Addition of two fractions
    def __add__(self, other):
        other = other if isinstance(other, Fraction) else Fraction(other)
        return Fraction(self.num * other.den + other.num * self.den, self.den * other.den)

    # So that 0 + Fraction also works (needed for sum())
    __radd__ = __add__

    # Subtraction
    def __sub__(self, other):
        return Fraction(self.num * other.den - other.num * self.den, self.den * other.den)

    # Multiplication
    def __mul__(self, other):
        other = other if isinstance(other, Fraction) else Fraction(other)
        return Fraction(self.num * other.num, self.den * other.den)

    __rmul__ = __mul__

    # Division
    def __truediv__(self, other):
        return Fraction(self.num * other.den, self.den * other.num)

    # Convert to decimal
    def to_decimal(self):
        return self.num / self.den

In [2]:
# Create two fraction objects
f1 = Fraction(2, 4)
f2 = Fraction(4, 6)

print("Fraction 1:", f1)
print("Fraction 2:", f2)
print("Addition:", f1 + f2)
print("Subtraction:", f1 - f2)
print("Multiplication:", f1 * f2)
print("Division:", f1 / f2)
print("Number times fraction:", 3 * f1)
print("Sum of six 1/6:", sum([Fraction(1, 6)] * 6))
print("Decimal of Fraction 2:", f2.to_decimal())

Fraction 1: 1/2
Fraction 2: 2/3
Addition: 7/6
Subtraction: -1/6
Multiplication: 1/3
Division: 3/4
Number times fraction: 3/2
Sum of six 1/6: 1
Decimal of Fraction 2: 0.6666666666666666


- `Fraction(2, 4)` is stored as `1/2`, because the constructor divides both parts by their **GCD** (greatest common divisor).
- `3 * f1`: Python first tries `int * Fraction`, which does not work, so it calls `f1.__rmul__(3)`.
- `sum()` starts from `0`, so it needs `0 + Fraction`, which uses `__radd__`.

## **Example: A 2D Vector Class**
A **vector** has a size and a direction, like wind speed or the movement of a boat.

In [3]:
import math

class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, k):
        return Vector(self.x * k, self.y * k)

    __rmul__ = __mul__

    def __abs__(self):                   # Length of the vector
        return math.hypot(self.x, self.y)

# A boat crosses the Hooghly at 4 km/h, and the current pushes it downstream at 3 km/h
boat = Vector(4, 0)
current = Vector(0, -3)
velocity = boat + current
position_after_half_hour = velocity * 0.5

print("Velocity:", velocity, "| Speed:", abs(velocity), "km/h")
print("After 30 minutes:", position_after_half_hour, "->", abs(position_after_half_hour), "km from the ghat")

Velocity: Vector(4, -3) | Speed: 5.0 km/h
After 30 minutes: Vector(2.0, -1.5) -> 2.5 km from the ghat


## **Example from Remote Sensing: A Band Class That Supports Maths**
In remote sensing, we do **band maths** all the time, like `(NIR - Red) / (NIR + Red)` for NDVI.

If our `Band` class supports `+`, `-` and `/`, we can write the NDVI formula **exactly as it looks on paper**. (On Day 81, we will see that the NumPy library works in the same way.)

In [4]:
class Band:
    """A tiny satellite band: a name and a list of pixel values."""

    def __init__(self, name, values):
        self.name = name
        self.values = list(values)

    def __repr__(self):
        return f"Band({self.name}: {[round(v, 3) for v in self.values]})"

    def _combine(self, other, op, symbol):
        return Band(f"({self.name}{symbol}{other.name})",
                    [op(a, b) for a, b in zip(self.values, other.values)])

    def __add__(self, other):
        return self._combine(other, lambda a, b: a + b, "+")

    def __sub__(self, other):
        return self._combine(other, lambda a, b: a - b, "-")

    def __truediv__(self, other):
        return self._combine(other, lambda a, b: a / b if b != 0 else 0, "/")

    def mean(self):
        return sum(self.values) / len(self.values)

red = Band("Red", [0.05, 0.10, 0.20, 0.06])
nir = Band("NIR", [0.45, 0.30, 0.25, 0.02])

ndvi = (nir - red) / (nir + red)       # Looks exactly like the formula!
print(ndvi)
print("Average NDVI:", round(ndvi.mean(), 3))

Band(((NIR-Red)/(NIR+Red)): [0.8, 0.5, 0.111, -0.5])
Average NDVI: 0.228


Let us understand the code above:
- `nir - red` calls `nir.__sub__(red)`, which subtracts the pixels one by one and returns a **new Band**.
- `(nir - red) / (nir + red)` works because each part is a Band, and Band knows how to divide.
- The name of the result even shows the formula that was used.

In [5]:
# Quick question
# What will be printed?
# class N:
#     def __init__(self, v): self.v = v
#     def __add__(self, o): return N(self.v + o.v)
# print((N(2) + N(3)).v)